# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a structured object

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets and fields, and their `@id` values.

The Croissant schema organizes tabular data in **record sets**, each with one or more **fields**. Each entity (record set, field, column) has a unique `@id` value. Let's inspect the record sets and their fields.

In [ ]:
record_sets_info = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for record_set in metadata.record_sets:
        print(f"RecordSet name: {record_set.name} | @id: {record_set.id}")
        if hasattr(record_set, 'fields'):
            for field in record_set.fields:
                print(f"  Field name: {field.name} | @id: {field.id} | Data type: {getattr(field, 'data_type', 'unknown')}")
        record_sets_info.append(record_set)
else:
    print("No record sets found in metadata. The dataset might contain only a single table or is not following Croissant 1.0+ table convention.")

## 3. Data Extraction
Let's load the data from the available record set(s) into a DataFrame for analysis.

In Croissant, entities are identified by `@id`. We use these IDs to specify exactly which record set(s) we want to extract records for.

In [ ]:
# Find record_set @id(s) if available. Here we'll auto-detect the first one if possible.
record_set_ids = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for record_set in metadata.record_sets:
        record_set_ids.append(record_set.id)
else:
    # Fallback if record_sets list is empty or missing (try the default Croissant behavior)
    # Try to list records without specifying the record_set
    print("Trying to infer record set(s) automatically...")
    try:
        sample_records = list(dataset.records())  # get up to 5 sample records
        if sample_records:
            print(f"Found records without specifying record set. Columns: {list(sample_records[0].keys())}")
        else:
            print("No records found.")
    except Exception as e:
        print(f"Failed to read sample records: {e}")

# If no record sets, raise informative error in later sections
dataframes = {}
if record_set_ids:
    for rs_id in record_set_ids:
        print(f"Loading records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)

    # Print column names for the first record set
    first_rs = record_set_ids[0]
    print(f"\nColumns in {first_rs}:\n{dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())
else:
    print("No record sets found -- cannot extract tabular data.")

## 4. Exploratory Data Analysis (EDA)

We now process the loaded data by applying standard preprocessing and analysis operations. We'll:
- Filter records based on a numeric field,
- Normalize this field,
- Group by a string/categorical field.

> **All field references use their Croissant `@id`.**

Below we attempt to select a numeric field for demonstration:

In [ ]:
import numpy as np

if record_set_ids:
    # Pick the first record set for demonstration
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Attempt to auto-detect a numeric field by checking field @id and dtype
    candidate_numeric_fields = []
    if hasattr(metadata, 'record_sets') and metadata.record_sets:
        for rs in metadata.record_sets:
            if rs.id == rs_id:
                for fld in rs.fields:
                    # Accept field if its data_type is Number/Integer/Float or if dtype in DataFrame is numeric
                    if hasattr(fld, 'data_type') and fld.data_type in ['schema:Number', 'schema:Integer', 'schema:Float']:
                        candidate_numeric_fields.append(fld.id)
    if not candidate_numeric_fields:
        # Fallback: NumPy numeric types in DataFrame
        candidate_numeric_fields = df.select_dtypes(include=np.number).columns.tolist()

    if candidate_numeric_fields:
        # Use the first found numeric field
        numeric_field_id = candidate_numeric_fields[0]

        print(f"Using numeric field: {numeric_field_id}")
        # Choose an arbitrary threshold
        threshold = np.nanpercentile(df[numeric_field_id], 75)  # upper quartile

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field (z-score)
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, col_norm]].head())

        # Try to group by a categorical field (e.g., group by a sample/label column)
        # Find first non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field} (using Croissant field @id):")
            print(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")
    else:
        print("No numeric fields detected in record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization

Let's visualize the distribution of our selected numeric field and its relationship with a group field, if available.

Feel free to modify field `@id`s to explore other attributes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and 'numeric_field_id' in locals():
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='steelblue')
        plt.title(f"Distribution of field {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: no suitable numeric field detected.")

## 6. Conclusion

In this notebook, we:
- Loaded a FAIR-compliant Croissant dataset describing protein abundance and related metadata from mass spectrometry analysis of extracellular vesicles.
- Explored the dataset's structure via record set and field `@id`s.
- Extracted tabular data, filtered and normalized numeric fields, and performed groupby analyses based on Croissant field identifiers.
- Visualized key data attributes.

This workflow can be generalized to any Croissant dataset identified by `@id`s for robust programmatic access and reproducible analyses.